In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / 'src'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from credit_default.data import load_credit_data
from credit_default.features import clean_credit_data, split_features_target, get_feature_groups

from sklearn.model_selection import train_test_split

from credit_default.data import load_credit_data
from credit_default.features import (
    clean_credit_data,
    split_features_target,
    get_feature_groups,
    add_credit_behaviour_features,
    check_for_vif
)

from credit_default.modeling.pipelines import build_logistic_regression_pipeline

from credit_default.evaluation import (
    evaluate_classifier,
    threshold_report,
    add_business_utility
)


In [2]:
raw_data = load_credit_data()
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_AMT3        

In [3]:
feature_groups = get_feature_groups()
feature_groups

FeatureGroups(categorical_cols=['SEX', 'EDUCATION', 'MARRIAGE_CLEAN', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'], numeric_cols=['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'], bill_cols=['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6'], pay_amount_cols=['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'], payment_status_cols=['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'])

In [4]:
clean_data = clean_credit_data(raw_data)
clean_data

,LIMIT_BAL,SEX,EDUCATION,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,...,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month,MARRIAGE_CLEAN
0,20000,2,2,24,2,2,-1,-1,-2,-2,...,0,0,0,689,0,0,0,0,1,1
1,120000,2,2,26,-1,2,0,0,0,2,...,3455,3261,0,1000,1000,1000,0,2000,1,2
2,90000,2,2,34,0,0,0,0,0,0,...,14948,15549,1518,1500,1000,1000,1000,5000,0,2
3,50000,2,2,37,0,0,0,0,0,0,...,28959,29547,2000,2019,1200,1100,1069,1000,0,1
4,50000,1,2,57,-1,0,-1,0,0,0,...,19146,19131,2000,36681,10000,9000,689,679,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,39,0,0,0,0,0,0,...,31237,15980,8500,20000,5003,3047,5000,1000,0,1
29996,150000,1,3,43,-1,-1,-1,-1,0,0,...,5190,0,1837,3526,8998,129,0,0,0,2
29997,30000,1,2,37,4,3,2,-1,0,0,...,20582,19357,0,0,22000,4200,2000,3100,1,2
29998,80000,1,3,41,1,-1,0,0,0,-1,...,11855,48944,85900,3409,1178,1926,52964,1804,1,1


In [5]:
fe_clean_data, columnas_anadidas = add_credit_behaviour_features(clean_data)
fe_clean_data

,LIMIT_BAL,SEX,EDUCATION,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,...,default payment next month,MARRIAGE_CLEAN,BILL_AMT_mean,BILL_AMT_max,BILL_AMT_std,PAY_AMT_mean,PAY_AMT_max,PAY_AMT_std,debt_to_limit,payment_to_debt
0,20000,2,2,24,2,2,-1,-1,-2,-2,...,1,1,1284.000000,3913,1761.633219,114.833333,689,281.283072,0.064200,0.089434
1,120000,2,2,26,-1,2,0,0,0,2,...,1,2,2846.166667,3455,637.967841,833.333333,2000,752.772653,0.023718,0.292791
2,90000,2,2,34,0,0,0,0,0,0,...,0,2,16942.166667,29239,6064.518593,1836.333333,5000,1569.815488,0.188246,0.108388
3,50000,2,2,37,0,0,0,0,0,0,...,0,1,38555.666667,49291,10565.793518,1398.000000,2019,478.058155,0.771113,0.036259
4,50000,1,2,57,-1,0,-1,0,0,0,...,0,1,18223.166667,35835,10668.590074,9841.500000,36681,13786.230736,0.364463,0.540054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,39,0,0,0,0,0,0,...,0,1,120891.500000,208365,86697.439530,7091.666667,20000,6794.318234,0.549507,0.058661
29996,150000,1,3,43,-1,-1,-1,-1,0,0,...,0,2,3530.333333,8979,3200.534247,2415.000000,8998,3515.523859,0.023536,0.684071
29997,30000,1,2,37,4,3,2,-1,0,0,...,1,2,11749.333333,20878,9354.149660,5216.666667,22000,8390.093365,0.391644,0.443997
29998,80000,1,3,41,1,-1,0,0,0,-1,...,1,1,44435.166667,78379,32992.487323,24530.166667,85900,36314.167188,0.555440,0.552044


In [7]:
feature_groups.categorical_cols

['SEX',
 'EDUCATION',
 'MARRIAGE_CLEAN',
 'PAY_0',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6']

In [6]:
feature_groups.numeric_cols

['LIMIT_BAL',
 'AGE',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6']

In [8]:
columnas_anadidas

['BILL_AMT_mean',
 'BILL_AMT_max',
 'BILL_AMT_std',
 'PAY_AMT_mean',
 'PAY_AMT_max',
 'PAY_AMT_std',
 'debt_to_limit',
 'payment_to_debt']

In [9]:
fe_numeric_cols = feature_groups.numeric_cols.copy()
og_numeric_cols = feature_groups.numeric_cols.copy()
fe_numeric_cols.extend(columnas_anadidas)

In [10]:
fe_numeric_cols

['LIMIT_BAL',
 'AGE',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'BILL_AMT_mean',
 'BILL_AMT_max',
 'BILL_AMT_std',
 'PAY_AMT_mean',
 'PAY_AMT_max',
 'PAY_AMT_std',
 'debt_to_limit',
 'payment_to_debt']

In [11]:
vif_data =check_for_vif(fe_clean_data, fe_numeric_cols)

/Users/marcoantoniogarciamartinez/Documents/Python Local/Entorno_DS/Python_Versions/3.11.15/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [12]:
vif_data

,variable,VIF
11,PAY_AMT3,inf
18,PAY_AMT_mean,inf
15,BILL_AMT_mean,inf
14,PAY_AMT6,inf
13,PAY_AMT5,inf
12,PAY_AMT4,inf
10,PAY_AMT2,inf
9,PAY_AMT1,inf
8,BILL_AMT6,inf
7,BILL_AMT5,inf


In [13]:
vif_data =check_for_vif(clean_data, og_numeric_cols)

In [14]:
vif_data

,variable,VIF
4,BILL_AMT2,25.757112
7,BILL_AMT5,24.884761
5,BILL_AMT3,21.728054
6,BILL_AMT4,20.246033
0,const,16.459243
8,BILL_AMT6,14.926490
3,BILL_AMT1,13.913438
10,PAY_AMT2,2.221902
11,PAY_AMT3,1.734998
9,PAY_AMT1,1.691422


In [15]:
vif_data.to_json()

'{"variable":{"4":"BILL_AMT2","7":"BILL_AMT5","5":"BILL_AMT3","6":"BILL_AMT4","0":"const","8":"BILL_AMT6","3":"BILL_AMT1","10":"PAY_AMT2","11":"PAY_AMT3","9":"PAY_AMT1","13":"PAY_AMT5","12":"PAY_AMT4","1":"LIMIT_BAL","14":"PAY_AMT6","2":"AGE"},"VIF":{"4":25.7571119595,"7":24.8847613586,"5":21.7280544773,"6":20.246033168,"0":16.4592432669,"8":14.9264903269,"3":13.913437747,"10":2.2219020897,"11":1.7349978192,"9":1.6914221607,"13":1.6789005067,"12":1.6293895087,"1":1.2308230283,"14":1.1690726006,"2":1.0221981909}}'

In [16]:
bill_to_convine = [f'BILL_AMT{i}' for i in range(2, 7)]
bill_to_convine

['BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']

In [17]:
for variable in bill_to_convine:
    fe_numeric_cols.remove(variable)

In [18]:
vif_data =check_for_vif(fe_clean_data, fe_numeric_cols)
vif_data

/Users/marcoantoniogarciamartinez/Documents/Python Local/Entorno_DS/Python_Versions/3.11.15/lib/python3.11/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,variable,VIF
9,PAY_AMT6,inf
4,PAY_AMT1,inf
5,PAY_AMT2,inf
6,PAY_AMT3,inf
7,PAY_AMT4,inf
8,PAY_AMT5,inf
13,PAY_AMT_mean,inf
14,PAY_AMT_max,216.688374
15,PAY_AMT_std,190.379787
11,BILL_AMT_max,69.278650


In [19]:
pay_to_remove = [f'PAY_AMT{i}' for i in range(1, 7)]

for variable in pay_to_remove:
    fe_numeric_cols.remove(variable)

In [20]:
vif_data = check_for_vif(fe_clean_data, fe_numeric_cols)
vif_data

,variable,VIF
8,PAY_AMT_max,214.213097
9,PAY_AMT_std,188.685214
5,BILL_AMT_max,65.182552
4,BILL_AMT_mean,42.710496
0,const,21.089888
3,BILL_AMT1,15.859411
6,BILL_AMT_std,9.185769
7,PAY_AMT_mean,7.194859
10,debt_to_limit,2.719941
1,LIMIT_BAL,2.224398


In [21]:
vif_data.to_json()

'{"variable":{"8":"PAY_AMT_max","9":"PAY_AMT_std","5":"BILL_AMT_max","4":"BILL_AMT_mean","0":"const","3":"BILL_AMT1","6":"BILL_AMT_std","7":"PAY_AMT_mean","10":"debt_to_limit","1":"LIMIT_BAL","2":"AGE","11":"payment_to_debt"},"VIF":{"8":214.2130973647,"9":188.6852140647,"5":65.1825517019,"4":42.7104963529,"0":21.0898884448,"3":15.8594112658,"6":9.1857693885,"7":7.1948594129,"10":2.7199410724,"1":2.2243979652,"2":1.0223684573,"11":1.0145165602}}'

In [22]:
candidatas_preliminares = ['LIMIT_BAL', 'AGE', 'BILL_AMT_mean', 'PAY_AMT_mean', 'debt_to_limit', 'payment_to_debt']

In [23]:
vif_data = check_for_vif(fe_clean_data, candidatas_preliminares)
vif_data

,variable,VIF
0,const,21.050038
5,debt_to_limit,2.714384
3,BILL_AMT_mean,2.641490
1,LIMIT_BAL,2.200730
4,PAY_AMT_mean,1.231373
2,AGE,1.021834
6,payment_to_debt,1.007456


In [24]:
vif_data.to_json()

'{"variable":{"0":"const","5":"debt_to_limit","3":"BILL_AMT_mean","1":"LIMIT_BAL","4":"PAY_AMT_mean","2":"AGE","6":"payment_to_debt"},"VIF":{"0":21.0500377124,"5":2.7143835592,"3":2.6414895949,"1":2.2007297188,"4":1.2313730029,"2":1.0218338443,"6":1.0074562944}}'

In [26]:
target_actual = 'default payment next month'

In [27]:
exporter_candidatos = candidatas_preliminares.copy()
exporter_candidatos.append(target_actual)

for variable in feature_groups.categorical_cols:
    exporter_candidatos.append(variable)

In [28]:
df_vif = fe_clean_data[exporter_candidatos]
df_vif

,LIMIT_BAL,AGE,BILL_AMT_mean,PAY_AMT_mean,debt_to_limit,payment_to_debt,default payment next month,SEX,EDUCATION,MARRIAGE_CLEAN,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
0,20000,24,1284.000000,114.833333,0.064200,0.089434,1,2,2,1,2,2,-1,-1,-2,-2
1,120000,26,2846.166667,833.333333,0.023718,0.292791,1,2,2,2,-1,2,0,0,0,2
2,90000,34,16942.166667,1836.333333,0.188246,0.108388,0,2,2,2,0,0,0,0,0,0
3,50000,37,38555.666667,1398.000000,0.771113,0.036259,0,2,2,1,0,0,0,0,0,0
4,50000,57,18223.166667,9841.500000,0.364463,0.540054,0,1,2,1,-1,0,-1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,39,120891.500000,7091.666667,0.549507,0.058661,0,1,3,1,0,0,0,0,0,0
29996,150000,43,3530.333333,2415.000000,0.023536,0.684071,0,1,3,2,-1,-1,-1,-1,0,0
29997,30000,37,11749.333333,5216.666667,0.391644,0.443997,1,1,2,2,4,3,2,-1,0,0
29998,80000,41,44435.166667,24530.166667,0.555440,0.552044,1,1,3,1,1,-1,0,0,0,-1


In [29]:
df_vif.info()

<class 'pandas.DataFrame'>
Index: 29986 entries, 0 to 29999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   LIMIT_BAL                   29986 non-null  int64  
 1   AGE                         29986 non-null  int64  
 2   BILL_AMT_mean               29986 non-null  float64
 3   PAY_AMT_mean                29986 non-null  float64
 4   debt_to_limit               29986 non-null  float64
 5   payment_to_debt             29986 non-null  float64
 6   default payment next month  29986 non-null  int64  
 7   SEX                         29986 non-null  int64  
 8   EDUCATION                   29986 non-null  int64  
 9   MARRIAGE_CLEAN              29986 non-null  int64  
 10  PAY_0                       29986 non-null  int64  
 11  PAY_2                       29986 non-null  int64  
 12  PAY_3                       29986 non-null  int64  
 13  PAY_4                       29986 non-null  int

In [30]:
X, y = split_features_target(df_vif)

In [31]:
X

,LIMIT_BAL,AGE,BILL_AMT_mean,PAY_AMT_mean,debt_to_limit,payment_to_debt,SEX,EDUCATION,MARRIAGE_CLEAN,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6
0,20000,24,1284.000000,114.833333,0.064200,0.089434,2,2,1,2,2,-1,-1,-2,-2
1,120000,26,2846.166667,833.333333,0.023718,0.292791,2,2,2,-1,2,0,0,0,2
2,90000,34,16942.166667,1836.333333,0.188246,0.108388,2,2,2,0,0,0,0,0,0
3,50000,37,38555.666667,1398.000000,0.771113,0.036259,2,2,1,0,0,0,0,0,0
4,50000,57,18223.166667,9841.500000,0.364463,0.540054,1,2,1,-1,0,-1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,39,120891.500000,7091.666667,0.549507,0.058661,1,3,1,0,0,0,0,0,0
29996,150000,43,3530.333333,2415.000000,0.023536,0.684071,1,3,2,-1,-1,-1,-1,0,0
29997,30000,37,11749.333333,5216.666667,0.391644,0.443997,1,2,2,4,3,2,-1,0,0
29998,80000,41,44435.166667,24530.166667,0.555440,0.552044,1,3,1,1,-1,0,0,0,-1


In [32]:
y

0        1
1        1
2        0
3        0
4        0
        ..
29995    0
29996    0
29997    1
29998    1
29999    1
Name: default payment next month, Length: 29986, dtype: int64

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    stratify = y,
    random_state = 42
)

In [34]:
model = build_logistic_regression_pipeline(
    numeric_cols = candidatas_preliminares,
    categorical_cols =feature_groups.categorical_cols
)

In [35]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [36]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [40]:
y_proba = model.predict_proba(X_test)[:, 1]

evaluate_classifier(
    y_true = y_test,
    y_proba = y_proba,
    threshold = 0.5
    
    )

{'threshold': 0.5,
 'accuracy': 0.768756252084028,
 'precision': 0.4803664921465969,
 'recall': 0.5531273549359458,
 'f1': 0.5141856392294221,
 'roc_auc': 0.7575332540550274,
 'pr_auc': 0.5296171573319789,
 'tn': 3877,
 'fp': 794,
 'fn': 593,
 'tp': 734}

In [41]:
threshold_df = threshold_report(
    y_true = y_test,
    y_proba = y_proba
)

In [39]:
threshold_df

,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,0.05,0.222574,0.221536,1.000000,0.362717,0.757533,0.529617,8,4663,0,1327
1,0.10,0.225909,0.222092,0.998493,0.363362,0.757533,0.529617,30,4641,2,1325
2,0.15,0.240247,0.224966,0.995479,0.366995,0.757533,0.529617,120,4551,6,1321
3,0.20,0.273424,0.231627,0.985682,0.375108,0.757533,0.529617,332,4339,19,1308
4,0.25,0.335112,0.245554,0.967596,0.391702,0.757533,0.529617,726,3945,43,1284
5,0.30,0.425975,0.267575,0.917860,0.414356,0.757533,0.529617,1337,3334,109,1218
6,0.35,0.556686,0.309169,0.813112,0.447997,0.757533,0.529617,2260,2411,248,1079
7,0.40,0.672724,0.372800,0.702336,0.487066,0.757533,0.529617,3103,1568,395,932
8,0.45,0.730744,0.424290,0.608139,0.499845,0.757533,0.529617,3576,1095,520,807
9,0.50,0.768756,0.480366,0.553127,0.514186,0.757533,0.529617,3877,794,593,734
